# Analysis of pp Events: Event Activity Observables

Firstly, we load libraries and enable multi-threading, which allows event processing across multiple CPU cores.

In [12]:
import ROOT
import numpy as np
import os

ROOT.EnableImplicitMT() # enable multi-threading

We connect RDataFrame to the TTree in our created events.root and events_smeared.root files.

In [13]:
# we must select the data we want to work with by their seed
seed = 2

dfTrue = ROOT.RDataFrame("events", f"data/{seed}/events{seed}.root") # connect RDataFrame to the TTree in events.root
dfSmeared = ROOT.RDataFrame("events_smeared", f"data/{seed}/events{seed}_smeared.root") # connect RDataFrame to the TTree in events_smeared.root

# define a data frame where true observables are replaced with the smeared ones so that we can write simple for loops
dfOnlySmeared = (
    dfSmeared.Redefine("pT", "pTSmeared")
             .Redefine("eta", "etaSmeared")
             .Redefine("phi", "phiSmeared")
)                 

# create a data frame dictionary which we can loop over
dataFrames = {
    'True': dfTrue,
    'Smeared': dfOnlySmeared
}

We also make a file in which we will store the created histograms and response matrices for future unfolding, and a directory to store .pdf files in.

In [14]:
# create the directory for images
os.makedirs(f"img/{seed}", exist_ok=True)

# create the root file for histograms
file = ROOT.TFile(f"data/{seed}/events{seed}_plots.root", "RECREATE")

### Multiplicity Distribution

In [15]:
# loop over data frames
for label, df in dataFrames.items():
    dfNch = df.Define("Nch", "pT.size()") # define the multiplicity column

    # load the histogram of the multiplicity distribution from the data frame
    histoNch = dfNch.Histo1D((f"histo{seed}Nch{label}", label + " Multiplicity;N_{ch};Normalized Events", 70, 0, 70), "Nch").GetValue()
    histoNch.Write() # store the histogram into the .root file -- note that we must save the histogram before normalizing it, as our response matrix is not constructed from the normalized distributions (and unfolding would thus fail)
    
    # set up canvas for drawing histograms
    canvasNch = ROOT.TCanvas(f"canvas{seed}Nch{label}", f"{label} Multiplicity", 800, 600) # create a canvas
    canvasNch.SetLogy() # set logarithmic scale for y-axis

    # normalize the histogram
    integral = histoNch.Integral()
    if integral > 0: # check if the histogram has entries to avoid division by zero
        histoNch.Scale(1.0 / integral) # normalize the histogram

    # customize the histogram
    histoNch.SetLineColor(ROOT.kRed + 1) # set line color to slightly darker (+1) red
    histoNch.SetFillColorAlpha(ROOT.kRed, 0.1) # set fill color to red with 10% transparency (alpha = 0.1)

    # draw and save
    histoNch.Draw("HIST") # draw the histogram
    canvasNch.SaveAs(f"img/{seed}/{seed}Nch{label}.pdf") # save the canvas as a PDF file

Info in <TCanvas::Print>: pdf file img/2/2NchTrue.pdf has been created
Info in <TCanvas::Print>: pdf file img/2/2NchSmeared.pdf has been created


### Multiplicity Response Matrix

In [16]:
dfNchRM = (
    dfSmeared.Define("NchTrue", "pT.size()") # define true multiplicity
             .Define("NchSmeared", "pTSmeared.size()") # define smeared multiplicity
)

# load the 2D histogram from the data frame
histoNchRM = dfNchRM.Histo2D((f"histo{seed}NchRM", "Multiplicity Response Matrix;N_{ch}^{MC};N_{ch}^{sm}", 70, 0, 70, 70, 0, 70), "NchTrue", "NchSmeared").GetValue() # 2D histogram with true Nch on the x-axis and smeared Nch on the y-axis

# set up canvas for drawing
canvasNchRM = ROOT.TCanvas(f"canvas{seed}NchRM", "Multiplicity Response Matrix", 800, 600)
canvasNchRM.SetLogz()

# draw and save
histoNchRM.Draw("COLZ")
canvasNchRM.SaveAs(f"img/{seed}/{seed}NchRM.pdf")

Info in <TCanvas::Print>: pdf file img/2/2NchRM.pdf has been created


As RooUnfold (which we are going to use for Bayesian unfolding) requires a response matrix with swapped axes, we store this histogram in the .root file. 

In [17]:
histoNchRMUnfolding = dfNchRM.Histo2D((f"histo{seed}NchRM", "Multiplicity Response Matrix;N_{ch}^{sm};N_{ch}^{MC}", 70, 0, 70, 70, 0, 70), "NchSmeared", "NchTrue").GetValue() # 2D histogram with smeared Nch on the x-axis and true Nch on the y-axis
histoNchRMUnfolding.Write()

4148

### Unweighted Spherocity $S_0^{p_\mathrm{T} = 1}$ Distribution

calculated only for $N_\mathrm{ch}(|\eta| < 1) > 10$

$$
S_0^{p_\mathrm{T} = 1} = \frac{\pi^2}{4} \min_{\hat{n}} \left( \frac{\sum_i |\hat{u}_{\mathrm{T}, i} \times \hat{n}|}{N_\mathrm{trks}} \right)^2
$$

In [18]:
# I wrote the calc_S0pT1(px, py, pT, Nch) function in C++, now we need to tell ROOT to load it and compile it. 
ROOT.gInterpreter.ProcessLine('#include "cpp/S0pT1.cpp"')

# loop over data frames
for label, df in dataFrames.items():
    dfS0pT1 = (
        df.Define("px", "pT * cos(phi)")
          .Define("py", "pT * sin(phi)")
          .Define("Nch", "pT.size()")
          .Define("ux", "px / pT")
          .Define("uy", "py / pT")
          .Filter("Nch > 10", "multiplicity cut") # filter only events with multiplicity greater than 10
          .Define("S0pT1", "calc_S0pT1(ux, uy, Nch)") # define the unweighted transverse spherocity column
    )

    # load the histogram from the data frame
    histoS0pT1 = dfS0pT1.Histo1D((f"histo{seed}S0pT1{label}", label + " Unweighted Transverse Spherocity;S_{0}^{p_{T} = 1};Normalized Events", 200, 0, 1), "S0pT1").GetValue() # histogram of the unweighted spherocity distribution
    histoS0pT1.Write() # store the histogram into the .root file

    # set up canvas for drawing
    canvasS0pT1 = ROOT.TCanvas(f"canvas{seed}S0pT1{label}", f"{label} S0pT1", 800, 600) # create a canvas to draw the histogram
    canvasS0pT1.SetLogy() # set logarithmic scale for y-axis

    # normalize the histogram
    integral = histoS0pT1.Integral()
    if integral > 0: # check if the histogram has entries to avoid division by zero
        histoS0pT1.Scale(1.0 / integral) # normalize the histogram

    # customize the histogram
    histoS0pT1.SetLineColor(ROOT.kRed + 1) # set line color to slightly darker (+1) red
    histoS0pT1.SetFillColorAlpha(ROOT.kRed, 0.1) # set fill color to red with 10% transparency (alpha = 0.1)

    # draw and save 
    histoS0pT1.Draw("HIST") # draw the histogram
    canvasS0pT1.SaveAs(f"img/{seed}/{seed}S0pT1{label}.pdf") # save the canvas as a PDF file

In file included from input_line_179:1:
./cpp/S0pT1.cpp:6:8: error: redefinition of 'calc_S0pT1'
double calc_S0pT1(const ROOT::RVec<double>& uxEvent, // const is a safety lock to ensure that the function only reads data and does not change it
       ^
./cpp/S0pT1.cpp: note: previous definition is here
Info in <TCanvas::Print>: pdf file img/2/2S0pT1True.pdf has been created
Info in <TCanvas::Print>: pdf file img/2/2S0pT1Smeared.pdf has been created


### Unweighted Spehrocity Response Matrix

In [19]:
ROOT.gInterpreter.ProcessLine('#include "cpp/S0pT1.cpp"') # load the calc_S0pT1 function

dfS0pT1RM = ( # ted me nenapada, jak to zefektivnit, tak to zatim napisu takto
    dfSmeared.Define("px", "pT * cos(phi)")
             .Define("py", "pT * sin(phi)")
             .Define("NchTrue", "pT.size()")
             .Define("ux", "px / pT")
             .Define("uy", "py / pT")
             .Define("pxSmeared", "pTSmeared * cos(phiSmeared)")
             .Define("pySmeared", "pTSmeared * sin(phiSmeared)")
             .Define("NchSmeared", "pTSmeared.size()")
             .Define("uxSmeared", "pxSmeared / pTSmeared")
             .Define("uySmeared", "pySmeared / pTSmeared")
             .Filter("NchTrue > 10", "true multiplicity cut")
             .Filter("NchSmeared > 10", "smeared multiplicity cut")
             .Define("S0pT1True", "calc_S0pT1(ux, uy, NchTrue)")
             .Define("S0pT1Smeared", "calc_S0pT1(uxSmeared, uySmeared, NchSmeared)")
)

# load the 2D histogram from the data frame
histoS0pT1RM = dfS0pT1RM.Histo2D((f"histo{seed}S0pT1RM", "Unweighted Spherocity Response Matrix;S_{0}^{p_{T}=1, MC};S_{0}^{p_{T}=1, sm}", 200, 0, 1, 200, 0, 1), "S0pT1True", "S0pT1Smeared").GetValue() # 2D histogram with true unweighted spherocity on the x-axis and smeared unweighted spherocity on the y-axis

# set up canvas for drawing
canvasS0pT1RM = ROOT.TCanvas(f"canvas{seed}S0pT1RM", "Unweighted Spherocity Response Matrix", 800, 600)
canvasS0pT1RM.SetLogz()

# draw and save
histoS0pT1RM.Draw("COLZ")
canvasS0pT1RM.SaveAs(f"img/{seed}/{seed}S0pT1RM.pdf")

In file included from input_line_182:1:
./cpp/S0pT1.cpp:6:8: error: redefinition of 'calc_S0pT1'
double calc_S0pT1(const ROOT::RVec<double>& uxEvent, // const is a safety lock to ensure that the function only reads data and does not change it
       ^
./cpp/S0pT1.cpp: note: previous definition is here
Info in <TCanvas::Print>: pdf file img/2/2S0pT1RM.pdf has been created


Once again, we must construct and save a RooUnfold compatible response matrix.

In [20]:
histoS0pT1RMUnfolding = dfS0pT1RM.Histo2D((f"histo{seed}S0pT1RM", "Unweighted Spherocity Response Matrix;S_{0}^{p_{T}=1, rec};S_{0}^{p_{T}=1, sm}", 200, 0, 1, 200, 0, 1), "S0pT1Smeared", "S0pT1True").GetValue() # 2D histogram with true unweighted spherocity on the x-axis and smeared unweighted spherocity on the y-axis
histoS0pT1RMUnfolding.Write()

41323

### $R_\mathrm{T}$ Distribution

TODO

### $R_\mathrm{T}$ Response Matrix

TODO

### $p_\mathrm{T}$ Spectra

In this section, we prepare 2D histograms of $S_0^{p_\mathrm{T} = 1}$ vs $p_\mathrm{T}$ (2D spectra) for unfolding. 

> **Note:** We use particle-level (or true) $p_\mathrm{T}$. If we were unfolding real experimental data, we would have to use $p_\mathrm{T}$ corrected for reconstruction efficiency (smearing) and detector acceptance.


Mapped onto **multiplicity** classes:

Mapped onto **unweighted spherocity** classes: 

-- V PYTHII JSEM NASTAVOVAL pT CUT 0.2 GeV, ALE pT SE MI PAK ROZMAZAVA (VIZ NIZE) NA HODNOTY POD 0.2 -- NEMEL BYCH JE .Filter()?

In [21]:
# create 2D histograms of unweighted spherocity vs pT (0-10 GeV)
histopTS0pT1True = dfS0pT1RM.Histo2D((f"histo{seed}pTS0pT1True", "True Unweighted Spherocity vs p_{T};p_{T}^{MC} [GeV];S_{0}^{p_{T}=1, MC}", 50, 0, 10, 200, 0, 1), "pT", "S0pT1True")
histopTS0pT1Smeared = dfS0pT1RM.Histo2D((f"histo{seed}pTS0pT1Smeared", "Smeared Unweighted Spherocity vs p_{T};p_{T}^{sm} [GeV];S_{0}^{p_{T}=1, sm}", 50, 0, 10, 200, 0, 1), "pT", "S0pT1Smeared")

# dictionary for the for cycle
histospTS0pT1 = {
    "True": histopTS0pT1True,
    "Smeared": histopTS0pT1Smeared
}

for label, histo in histospTS0pT1.items():
    # save the histogram for unfolding
    histo.Write() 

    # set up canvas
    canvaspTS0pT1 = ROOT.TCanvas(f"canvas{seed}pTS0pT1{label}", label + " Unweighted Spherocity vs p_{T}", 800, 600)
    canvaspTS0pT1.SetLogz()

    # draw and save
    histo.Draw("COLZ")
    canvaspTS0pT1.SaveAs(f"img/{seed}/{seed}pTS0pT1{label}.pdf")

Info in <TCanvas::Print>: pdf file img/2/2pTS0pT1True.pdf has been created
Info in <TCanvas::Print>: pdf file img/2/2pTS0pT1Smeared.pdf has been created


In the end, we close the .root file.

In [22]:
file.Close()